# CSV/Excel Analysis Agent: demo

This notebook runs the agent from this repo with LlamaIndex and Groq.
Run it from the notebooks folder in Jupyter, or in Google Colab (see the Colab cell).

You need a free Groq API key from https://console.groq.com.

## 1. Setup

In [ ]:
# Google Colab only: clone your repo and move into it. Skip this cell when running locally.
# Replace YOUR_USERNAME with your GitHub username.
# !git clone https://github.com/YOUR_USERNAME/csv-excel-analysis-agent.git
# %cd csv-excel-analysis-agent

In [ ]:
from pathlib import Path
requirements = Path("requirements.txt") if Path("requirements.txt").exists() else Path("../requirements.txt")
%pip install -q -r {requirements}

Add the key as `GROQ_API_KEY` in Colab Secrets, in a `.env` file, or type it below.
The key is never written into the notebook.

In [ ]:
import os, sys, getpass
from pathlib import Path

# Make the repo importable (works from notebooks/ or from the repo root)
repo_root = Path.cwd() if (Path.cwd() / "csv_agent").exists() else Path.cwd().parent
sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

from csv_agent.config import read_api_key

if not read_api_key():
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
print("Key found:", bool(read_api_key()))

## 2. Build the agent

In [ ]:
from llama_index.core.workflow import Context
from csv_agent import build_agent
from csv_agent.runner import ask

agent, store = build_agent()
ctx = Context(agent)  # keeps the chat memory between questions
print("Tools:", [t.metadata.name for t in agent.tools])

## 3. Ask questions

The `[tool]` lines show which tool the agent chose and with what arguments.

In [ ]:
print(await ask(agent, ctx, "Load data/sample_sales.csv and give me a short overview."))

In [ ]:
print(await ask(agent, ctx, "Which region has the highest total revenue?"))

In [ ]:
print(await ask(agent, ctx, "Are there any data quality problems? Fix the duplicates and the extra spaces, and fill missing customer ages with the median."))

In [ ]:
print(await ask(agent, ctx, "Plot monthly revenue as a line chart."))

## 4. Look at the chart and the cleaned data

In [ ]:
from IPython.display import Image, display

charts = sorted(Path("outputs").glob("*.png"), key=lambda p: p.stat().st_mtime)
if charts:
    display(Image(filename=str(charts[-1])))
store.df.head()

## 5. Try your own file

Put any CSV or Excel file in the `data/` folder and ask about it.

In [ ]:
# print(await ask(agent, ctx, "Load data/my_file.xlsx and tell me what is in it."))